## Import libraries

In [ ]:
import numpy as np
import tomopy
from helperFunctions import MoviePlotter, convert_to_numpy, degree_to_positiveRadians
from tomoDataClass import tomoData

## Load Co-doped Data

Instead of a simulated Shepp3D phantom, this notebook uses the real Co-doped phase projections saved in the `tomoMono` folder:

- `Co_doped_phase_stack.tiff` — the 3D projection stack (angle, y, x)
- `Co_doped_angles.npy` — the tilt angle (degrees) for each slice, index-matched to the stack
- `Co_doped_slice_order.csv` — human-readable record of `slice_index, scan, angle_deg`

Because this is measured experimental data, it already contains real misalignment and noise, so the phantom-only `jitter()` / `add_noise()` steps are dropped.

In [ ]:
# File paths for the saved Co-doped data
PHASE_STACK = 'Co_doped_phase_stack.tiff'
ANGLES_FILE = 'Co_doped_angles.npy'
SLICE_ORDER = 'Co_doped_slice_order.csv'

algorithm = 'gridrec'

In [ ]:
# Load the phase projection stack (angle, y, x)
projections, scale_info = convert_to_numpy(PHASE_STACK)
projections = projections.astype(np.float32)
downsample = 1
projections = projections[:, 60:-262:downsample, 75:-87:downsample] #Crop to be good size 1000,600 at full res2
print(projections.shape)
print("Measured Co-doped phase projections")
MoviePlotter(projections)  # Plots each projection

In [ ]:
# Load the per-slice tilt angles (saved in degrees) and convert to positive radians
angles_deg = np.load(ANGLES_FILE)
angles = degree_to_positiveRadians(angles_deg)
angles = (angles + np.pi) % (2 * np.pi)
print(f"{len(angles)} angles, spanning {angles_deg.min()} to {angles_deg.max()} degrees")
assert len(angles) == projections.shape[0], "angle count must match number of projections"
print(angles)

In [ ]:
# croppedProjections = projections[::, 60:-60:4, 75:-75:4]
tomo = tomoData(projections, angles)
print("Measured projections (real misalignment + noise already present)")
tomo.makeNotebookProjMovie()

## Align Data

In [ ]:
#Show bad reconstruction prior to alignment
tomo.reconstruct(algorithm=algorithm, find_center_method='vo')
print("\nBad reconstruction prior to alignment")
badRecon = tomo.recon.copy()

tomo.makeNotebookReconMovie()

# print(tomo.ang)

In [ ]:
combined_rmse, x_rmse, y_rmse, x_cm, y_cm = tomo.sinogram_consistency_score(plot=True)

In [ ]:
# tomo.reset_workingProjections() #You can pass x_size/y_size for tighter cropping
# tomo.normalize(isPhaseData=True)
# tomo.cross_correlate_align(tolerance=0.001, max_iterations=20, stepRatio=0.9, yROI_Range=None, xROI_Range=None)
# tomo.projection_matching_alignment(max_iterations=5, tolerance=0.05, algorithm='gridrec', standardize=False)
# tomo.make_updates_shift()

tomo.reset_workingProjections() #You can pass x_size/y_size for tighter cropping
tomo.normalize(isPhaseData=True)
# tomo.center_projections(method='vo')

tomo.cross_correlate_align(
        tolerance=0, maxShiftTolerance=0, max_iterations=5, stepRatio=0.8,
        downsample=4, use_grad=False,
        yROI_Range=None,
        xROI_Range=None,
        isFull360=False,
        plot=True
    )
tomo.center_projections(method='vo')
tomo.cross_correlate_align(
        tolerance=0, maxShiftTolerance=0, max_iterations=5, stepRatio=0.8,
        downsample=2, use_grad=False,
        yROI_Range=[300//downsample, 750//downsample],
        xROI_Range=[70//downsample,530//downsample],
        # yROI_Range=None,
        # xROI_Range=None,
        isFull360=False,
        plot=True
    )
tomo.cross_correlate_align(
        tolerance=0, maxShiftTolerance=0, max_iterations=5, stepRatio=0.8,
        downsample=1, use_grad=False,
        yROI_Range=[0, 300//downsample],
        xROI_Range=[140//downsample,460//downsample],
        # yROI_Range=None,
        # xROI_Range=None,
        isFull360=False,
        plot=True
    )
tomo.center_projections(method='vo')

tomo.make_updates_shift()

MoviePlotter(tomo.workingProjections)  # Plots each projection
# tomo.makeNotebookProjMovie()

In [ ]:
import matplotlib.pyplot as plt
def displayWorkingSinogram(row_index=None):
    if row_index is None:
        row_index = tomo.workingProjections.shape[1] // 2
    plt.imshow(tomo.workingProjections[:,row_index,:], cmap='gray')
    plt.title(f'Sinogram (Row {row_index})')
    plt.ylabel('Angle')
    plt.show()
displayWorkingSinogram()

In [ ]:
tomo.projection_matching_alignment(
        levels=1, scale=1, iterations_per_level=[3],
        tolerance=0.0, algorithm=algorithm, standardize=False, use_grad=True,
        shift_method='cross_correlation', of_sigma=2.0, stepRatio=0.8, centering_method='vo',
        plot=True,
        yROI_Range=[0, 300//downsample],
        xROI_Range=[140//downsample, 460//downsample],
        # xROI_Range=None,
        # yROI_Range=None,
    )
tomo.make_updates_shift()
tomo.makeNotebookProjMovie()


In [ ]:
tomo.reconstruct(algorithm=algorithm, find_center_method='vo')
print("\nGood Reconstruction after alignment")
tomo.makeNotebookReconMovie()

# Save the reconstruction as a tiff file from helperFunctions.py save_to_tiff 
# def convert_to_tiff(numpy_data, file_location, scale_info=None):
from helperFunctions import convert_to_tiff
convert_to_tiff(-tomo.recon, f'Co_doped_reconstruction_{downsample}xds.tiff', scale_info=scale_info)

## Assess Alignment Quality

Now that alignment and reconstruction are done, we quantify how good the alignment is with two metrics:

- **Reprojection Consistency Score (RCS)** — *the most reliable metric for alignment quality.* It reprojects the reconstructed volume at every angle and compares it against the measured projections (per-angle NRMSE). Lower is better (`< 0.10` is excellent), and the per-angle bar chart flags any projections that are still misaligned.
- **Sinogram Consistency Score** — *a rough gauge, not a reliable score.* It checks the Helgason-Ludwig center-of-mass conditions on the sinogram. Use it to get a coarse sense of how close the alignment is, to spot outlier projections, and to view the central-slice sinogram.

(For *reconstruction* quality / resolution rather than alignment, use `fourier_shell_correlation` — the gold-standard half-dataset FSC metric.)

In [ ]:
# Reprojection Consistency Score — best metric of alignment quality (lower is better)
rcs, per_angle_nrmse, _ = tomo.reprojection_consistency_score(plot=True)

In [ ]:
# Sinogram Consistency Score — rough gauge of alignment + central-slice sinogram view
combined_rmse, x_rmse, y_rmse, x_cm, y_cm = tomo.sinogram_consistency_score(plot=True)